# Most Viewed Videos (Top 1000)

A focused, output-driven analysis of what dominates global high-view videos and where engagement differs.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but those won't be saved outside of the current session


## 1. Load and Normalize

Load the file and convert `views` / `likes` from human-readable text (`M`, `B`) into numeric fields.

In [ ]:
from pathlib import Path

csv_name = 'most_viewed_videos_1000.csv'
candidate_paths = list(Path('/kaggle/input').rglob(csv_name))
path = str(candidate_paths[0]) if candidate_paths else csv_name

df = pd.read_csv(path)
print('Loaded from:', path)
print('shape:', df.shape)
display(df.head())

def to_numeric_human(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    cleaned = series.astype(str).str.strip().str.replace(',', '', regex=False)
    mult = cleaned.str.extract(r'(?i)([KMB])$')[0].str.upper().map({'K':1e3, 'M':1e6, 'B':1e9}).fillna(1)
    nums = pd.to_numeric(cleaned.str.replace(r'(?i)[KMB]$', '', regex=True), errors='coerce')
    return nums * mult

df['views_num'] = to_numeric_human(df['views'])
df['likes_num'] = to_numeric_human(df['likes'])
df['like_rate'] = np.where(df['views_num'] > 0, df['likes_num'] / df['views_num'], np.nan)
df['is_short'] = df['is_short'].astype(int)
df['has_hashtags'] = df['has_hashtags'].astype(int)

profile = pd.DataFrame({
    'missing': df.isna().sum(),
    'dtype': df.dtypes.astype(str),
})
display(profile)


## 2. Quick Facts from Output

These are computed from the loaded dataset in this run.

In [ ]:
summary = {
    'rows': len(df),
    'median_views': df['views_num'].median(),
    'max_views': df['views_num'].max(),
    'median_likes': df['likes_num'].median(),
    'max_likes': df['likes_num'].max(),
    'views_likes_corr': df[['views_num', 'likes_num']].corr().iloc[0, 1],
    'short_share_pct': df['is_short'].mean() * 100,
    'hashtag_share_pct': df['has_hashtags'].mean() * 100,
}

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['value']
display(summary_df)

top_video = df.loc[df['views_num'].idxmax(), ['title', 'views_num', 'likes_num', 'content_type']]
print('Top video by views:', top_video['title'])
print('Views:', f"{top_video['views_num']:,.0f}", '| Likes:', f"{top_video['likes_num']:,.0f}", '| Type:', top_video['content_type'])


## 3. Content and Language Landscape

Check what kinds of videos and languages dominate this high-view segment.

In [ ]:
content_counts = df['content_type'].value_counts().rename('videos').to_frame()
lang_counts = df['detected_language'].value_counts().rename('videos').to_frame()

display(content_counts.head(10))
display(lang_counts.head(10))

content_perf = df.groupby('content_type', as_index=False).agg(
    videos=('title', 'count'),
    median_views=('views_num', 'median'),
    median_likes=('likes_num', 'median'),
    avg_like_rate=('like_rate', 'mean')
).sort_values('videos', ascending=False)
display(content_perf.head(10))


## 4. Visual Deep Dive

### 4.1 Top Videos by Views
One chart, larger canvas, cleaner labels.

In [ ]:
top_views = df[['title', 'views_num']].sort_values('views_num', ascending=False).head(12).sort_values('views_num').copy()
top_views['title_plot'] = top_views['title'].astype(str).str.encode('ascii', errors='ignore').str.decode('ascii')
top_views['title_plot'] = top_views['title_plot'].str.replace(r'\s+', ' ', regex=True).str.strip().replace('', 'Untitled')
top_views['title_plot'] = top_views['title_plot'].str.slice(0, 62)

plt.figure(figsize=(13, 8))
sns.barplot(data=top_views, y='title_plot', x='views_num', hue='title_plot', dodge=False, legend=False, palette='crest')
plt.title('Top Videos by Views')
plt.xlabel('Views (numeric)')
plt.ylabel('')
plt.tight_layout()
plt.show()


### 4.2 Distribution by Content Type

In [ ]:
top_content = df['content_type'].value_counts().head(8).index
tmp = df[df['content_type'].isin(top_content)].copy()

plt.figure(figsize=(12, 6))
sns.boxplot(data=tmp, x='content_type', y='likes_num', color='#86c5da')
plt.title('Likes Spread by Top Content Types')
plt.xlabel('Content Type')
plt.ylabel('Likes (numeric)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


### 4.3 Engagement Relationships

In [ ]:
sample = df.sample(min(500, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=sample, x='views_num', y='likes_num', hue='is_short', palette='Set2', alpha=0.7, s=55, ax=axes[0])
axes[0].set_title('Views vs Likes (Sampled)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Views (log)')
axes[0].set_ylabel('Likes (log)')

flag_perf = df.groupby('has_hashtags', as_index=False)['like_rate'].mean()
flag_perf['has_hashtags'] = flag_perf['has_hashtags'].map({0: 'No hashtags', 1: 'Has hashtags'})
sns.barplot(data=flag_perf, x='has_hashtags', y='like_rate', hue='has_hashtags', dodge=False, legend=False, ax=axes[1], palette='magma')
axes[1].set_title('Average Like Rate by Hashtag Use')
axes[1].set_xlabel('')
axes[1].set_ylabel('Average Like Rate')

plt.tight_layout()
plt.show()


## 5. Engagement Outliers

Highlight videos with unusually high like-rate while still having substantial view volume.

In [ ]:
high_exposure = df[df['views_num'] >= 2e8].copy()
outliers = high_exposure[['rank', 'title', 'content_type', 'views_num', 'likes_num', 'like_rate']]
outliers = outliers.sort_values('like_rate', ascending=False).head(15)
display(outliers)


## 6. Key Takeaways

- This dataset is large (`1000` rows) with no missing values, so comparisons are straightforward.
- `Music Video` and `Kids/Educational` are major contributors to the high-view list, but `Other` is also substantial.
- Views and likes are positively related, but not tightly enough to treat views as a direct proxy for engagement quality.
- Shorts and hashtagged titles show higher average like-rate in this dataset, though they are a smaller share of the total videos.